# Inference Configuration
Specify
- path

In [1]:
path = "../../output/protenn2/v5"


In [2]:
import json
import os.path
import pickle

import torch
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader

from src.protenn2.utils import get_train_val_test_paths, get_device

# get file paths

log_path = os.path.join(path, "log")

label_encoder_path = os.path.join(path, "label_encoder.pkl")
model_path = os.path.join(path, "best_model.pt")
with open(os.path.join(path, "params.json"), "r") as f:
    params = json.load(f)
if "input_folder" not in params:
    raise ValueError("input_folder must be specified")
dataset_path = os.path.join("../../", params["input_folder"])
train_path, val_path, test_path = get_train_val_test_paths(dataset_path)


In [3]:
from src.protenn2.utils import calculate_max_protein_length
from src.protenn2.dataset import CathPredPerResidueDataset, create_protein_collate_fn
from src.protenn2.model import CathPredEnn2
from src.protenn2.analysis.cath_hierarchy_mapper import CATHHierarchyMapper

# Initialize objects

device = get_device()
with open(label_encoder_path, "rb") as f:
    label_encoder: LabelEncoder = pickle.load(f)
num_classes = len(label_encoder.classes_)
max_protein_length = calculate_max_protein_length(dataset_path)

model = CathPredEnn2(num_classes=num_classes)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
val_dataset = CathPredPerResidueDataset(val_path, label_encoder=label_encoder,
                                        embedding_dir="../../data/embeddings/protein_embeddings_new")

collate_fn = create_protein_collate_fn(max_protein_length, val_dataset.padding_encoded_id)

val_dataloader = DataLoader(val_dataset, collate_fn=collate_fn)

mapper = CATHHierarchyMapper(label_encoder=label_encoder)

Using MPS (Apple Silicon GPU).
Max protein length: 599
Dataset initialized with 1319 unique proteins.


In [4]:
from src.protenn2.analysis.inference import run_inference

y_true_labels_list, y_pred_confidences_list = run_inference(model=model, dataloader=val_dataloader,
                                                            padding_encoded_id=val_dataset.padding_encoded_id,
                                                            device=device)

Running inference on 1319 proteins...


Inference Progress:  23%|██▎       | 306/1319 [00:00<00:02, 438.04it/s]

Inference Progress: 100%|██████████| 1319/1319 [00:03<00:00, 417.58it/s]

Inference complete. Processed 1319 proteins


# Analysis Configuration

In [5]:
from src.protenn2.utils import call_domains_list
bootstrap_samples = 1000
post_process_kwargs = {"reporting_threshold": 0.2, "region_min_length": 20, "gaussian_sigma": 1}
post_process_func = call_domains_list
# metrics_to_compute = ("accuracy", "f1_score", "jaccard_score", "recall_score", "precision_score",
#                                           "segment_overlap_score")

metrics_to_compute = ("segment_overlap_score")

In [ ]:
from src.protenn2.analysis.metrics import calculate_metrics_for_cath_levels

all_results = calculate_metrics_for_cath_levels(y_true_labels_list=y_true_labels_list,
                                                y_pred_confidences_list=y_pred_confidences_list, mapper=mapper,
                                                bootstrap_samples=bootstrap_samples,
                                                post_process_func=post_process_func,
                                                post_process_kwargs=post_process_kwargs,
                                                metrics_to_compute=metrics_to_compute)

---- Computing Metrics for hierarchy: C
----- Segment Overlap Score Metrics -----


Bootstrapping Progress:  49%|████▉     | 491/1000 [00:28<00:29, 17.23it/s]

# All results

In [10]:
all_results

{'raw_C': {'segment_overlap_score': {'mean': 52.31570649178702,
   'ci_lower': np.float64(50.10490967707039),
   'ci_upper': np.float64(54.58835823825208),
   'alpha': 0.05}},
 'post_C': {'segment_overlap_score': {'mean': 58.581111395995535,
   'ci_lower': np.float64(56.12673750042703),
   'ci_upper': np.float64(61.10441865908219),
   'alpha': 0.05}},
 'raw_A': {'segment_overlap_score': {'mean': 47.15254937798253,
   'ci_lower': np.float64(45.00352958255213),
   'ci_upper': np.float64(49.29291603506594),
   'alpha': 0.05}},
 'post_A': {'segment_overlap_score': {'mean': 56.405774780949805,
   'ci_lower': np.float64(53.98465375797438),
   'ci_upper': np.float64(58.78914690969676),
   'alpha': 0.05}},
 'raw_T': {'segment_overlap_score': {'mean': 44.9143044173385,
   'ci_lower': np.float64(42.78991345377491),
   'ci_upper': np.float64(47.09573348911323),
   'alpha': 0.05}},
 'post_T': {'segment_overlap_score': {'mean': 54.745599561270076,
   'ci_lower': np.float64(52.215042680893944),
   '

In [ ]:
with open(os.path.join(path, "sov_metrics.json"), "w") as f:
    json.dump(all_results, f)

In [ ]:
from src.protenn2.analysis.plot import plot_metric_with_ci

plt = plot_metric_with_ci(all_results, "accuracy")
plt.show()
plt = plot_metric_with_ci(all_results, "jaccard_score")
plt.show()
plt = plot_metric_with_ci(all_results, "f1_score")
plt.show()

In [ ]:
plt = plot_metric_with_ci(all_results, "segment_overlap_score")
plt.show()